# Critical evaluation of PINN for FWD inverse analysis and differentiable FEM as an alternative

**Paper:** Choi, Y., Moon, H., Ryu, S. (2026). *Critical evaluation of PINN for FWD inverse analysis and differentiable FEM as an alternative.* arXiv:2606.03210 [cs.CE].

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Critical evaluation of PINN for FWD inverse analysis and differentiable.pdf`

## Como se usan las PINNs en este paper

El paper evalua criticamente el uso de PINNs para el **backcalculo inverso** de los modulos elasticos $E_1,\dots,E_n$ de un pavimento multicapa a partir de la cuenca de deflexion medida por un deflectometro de impacto (FWD, Falling Weight Deflectometer). Comparan dos variantes de PINN:

1. **PINN vanilla**: una unica red $\mathcal{N}_\theta(r,z)\mapsto(u_r,u_z)$ para todo el dominio, con condiciones de contorno esenciales impuestas como *restricciones duras* mediante funciones de prueba (Eq. 9-10: $\hat u_r=r(L-r)z\,\mathcal{N}_\theta^{(r)}$, $\hat u_z=z\,\mathcal{N}_\theta^{(z)}$), y una perdida fisica basada en el residuo de equilibrio de la matriz de rigidez ensamblada por FEM: $\mathcal{L}_{phy}=\frac{1}{|\mathcal{I}_f|}\sum\|[\mathbf{K}(\mathbf{E})\hat{\mathbf{u}}-\mathbf{F}]_i\|^2$ (Eq. 12), con $\mathbf{K}(\mathbf{E})=\sum_i E_i\mathbf{K}^{(i)}$ (Eq. 6). Tanto $\mathbf{E}$ como $\theta$ son parametros entrenables, optimizados conjuntamente por diferenciacion automatica.
2. **XPINN** (PINN extendida con descomposicion de dominio, Jagtap & Karniadakis 2020): una **subred independiente por capa** $\mathcal{N}_{\theta_i}$, con continuidad de desplazamiento en las interfaces impuesta como termino de perdida blando (Eq. 14): $\mathcal{L}_{int}=\sum_i \frac{1}{|\Gamma_{i,i+1}|}\sum_j\|\mathcal{N}_{\theta_i}(x_j)-\mathcal{N}_{\theta_{i+1}}(x_j)\|^2$.

**Hallazgo clave del paper:** la PINN vanilla (una sola red) **no logra recuperar los modulos** debido a las discontinuidades de material entre capas (sesgo espectral); la XPINN mejora sustancialmente al asignar una subred por capa, pero sigue siendo sensible a la ponderacion de la perdida y a la arquitectura, y se degrada con ruido en las mediciones. El paper concluye que **DiffFEM** (FEM diferenciable, imponiendo la fisica como restriccion dura en vez de termino de perdida) es mas preciso y estable — pero ese metodo no usa una red neuronal, por lo que este cuaderno se centra en reproducir fielmente el **mecanismo PINN/XPINN**, que es el foco de "como se usan las PINNs" en el paper.

## Simplificacion declarada

El problema real es axisimetrico 2D $(r,z)$ con teoria elastica de capas de Burmister, resuelto en el paper mediante ensamblaje completo de una matriz de rigidez FEM. Reproducir ese solver FEM 2D esta fuera del alcance de este cuaderno. En su lugar, implementamos un **analogo 1D fiel al mismo mecanismo de inversion**: una barra elastica 1D con 3 capas de modulo $E_i$ distinto, en equilibrio estatico bajo una carga axial (equivalente a que el esfuerzo $\sigma=E(z)\,du/dz$ sea constante a lo largo de la barra, la version 1D de $\nabla\cdot\sigma=0$). Se conserva exactamente:
- La arquitectura de subredes por capa (XPINN) vs. red unica (vanilla PINN).
- Las restricciones duras de contorno via funciones de prueba (Eq. 9-10, adaptadas a 1D).
- La perdida de continuidad de interfaz blanda (Eq. 14).
- El aprendizaje conjunto de $\mathbf{E}$ (moduli, inicializados deliberadamente bajos como en el paper, $E=0.1$) y de los pesos de red $\theta$ por diferenciacion automatica.

## Repositorio publico de referencia

El PDF no incluye un repositorio propio (no hay seccion de codigo/datos disponibles en las paginas revisadas). El paper si cita explicitamente el metodo XPINN de Jagtap & Karniadakis (2020), cuya implementacion oficial esta en:

- **AmeyaJagtap/XPINNs** &mdash; https://github.com/AmeyaJagtap/XPINNs — implementacion original de Extended PINNs con descomposicion de dominio, el mismo mecanismo (subredes por subdominio + continuidad de interfaz) reproducido aqui.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Benchmark sintetico 1D (analogo al benchmark sintetico del paper, Seccion 4)

Barra elastica $\zeta\in[0,1]$ con base rigida en $\zeta=0$ ($u(0)=0$, restriccion dura como en Eq. 9-10) y carga axial $P_0$ aplicada en la superficie $\zeta=1$ (analogo del impulso FWD). Tres capas: subrasante $\Omega_1=[0,1/3]$, base $\Omega_2=[1/3,2/3]$, superficie $\Omega_3=[2/3,1]$, con modulos verdaderos $E_1=150$, $E_2=800$, $E_3=3000$ (MPa).

In [ ]:
E_true = [150.0, 800.0, 3000.0]
bounds = [0.0, 1/3, 2/3, 1.0]   # limites de Omega_1, Omega_2, Omega_3
P0 = 100.0                       # carga axial (esfuerzo) aplicada en la superficie

def analytic_u(zeta):
    """Solucion exacta: sigma = E_i * du/dz = P0 constante en toda la barra (equilibrio 1D).
    Se recorre capa a capa (sin recursion) para evitar solapar los limites de las mascaras."""
    zeta = np.atleast_1d(zeta).astype(float)
    u = np.zeros_like(zeta)
    u_left = 0.0
    for i in range(3):
        z0, z1 = bounds[i], bounds[i + 1]
        mask = (zeta >= z0) & (zeta < z1) if i < 2 else (zeta >= z0) & (zeta <= z1)
        u[mask] = u_left + (P0 / E_true[i]) * (zeta[mask] - z0)
        u_left = u_left + (P0 / E_true[i]) * (z1 - z0)
    return u

# Sensores (analogo a los geofonos del FWD): datos 'observados' en varias profundidades
sensor_zeta = np.array([0.05, 0.20, 0.40, 0.55, 0.75, 0.90, 1.00])
sensor_u = analytic_u(sensor_zeta)

zeta_plot = np.linspace(0, 1, 300)
plt.plot(zeta_plot, analytic_u(zeta_plot), label='Solucion analitica (verdad)')
plt.scatter(sensor_zeta, sensor_u, color='k', zorder=5, label='Sensores (datos observados)')
for b in bounds[1:-1]:
    plt.axvline(b, color='gray', linestyle=':')
plt.xlabel('$\\zeta$ (profundidad normalizada)')
plt.ylabel('$u(\\zeta)$')
plt.legend()
plt.title('Benchmark sintetico de 3 capas (analogo 1D del FWD)')
plt.show()

## 2. Arquitecturas: PINN vanilla (una red) vs. XPINN (una subred por capa, Eq. 9-10 y 13)

In [ ]:
class SubNet(nn.Module):
    """3 capas ocultas x 64 neuronas, tanh (misma arquitectura que el paper, Seccion 3.2.1)."""
    def __init__(self, n_hidden=3, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, zeta):
        return self.net(zeta)


class VanillaPINN(nn.Module):
    """Una unica red para todo el dominio; restriccion dura u(0)=0 (Eq. 9-10 adaptada a 1D)."""
    def __init__(self):
        super().__init__()
        self.net = SubNet()

    def forward(self, zeta):
        return zeta * self.net(zeta)


class XPINN(nn.Module):
    """Una subred por capa (Eq. 13); solo la subred de Omega_1 lleva la restriccion dura u(0)=0."""
    def __init__(self):
        super().__init__()
        self.nets = nn.ModuleList([SubNet() for _ in range(3)])

    def forward_domain(self, zeta, i):
        if i == 0:
            return zeta * self.nets[0](zeta)   # restriccion dura solo donde toca el contorno z=0
        return self.nets[i](zeta)


def d_dz(u, z):
    return torch.autograd.grad(u, z, grad_outputs=torch.ones_like(u),
                                create_graph=True, retain_graph=True)[0]

## 3. Funcion de perdida: residuo fisico + dato + (para XPINN) continuidad de interfaz (Eq. 11-15)

Para una barra 1D sin fuerza de cuerpo, el equilibrio $\nabla\cdot\sigma=0$ integra exactamente a $\sigma(z)=E(z)\,du/dz=P_0$ (constante) en toda la barra. Usamos esta forma integrada (equivalente a Eq. 12 + la condicion de carga en superficie) como residuo fisico: mas estable numericamente que diferenciar dos veces, y fisicamente exacta.

In [ ]:
n_col = 60
zeta_dom = [torch.linspace(bounds[i] + 1e-3, bounds[i + 1] - 1e-3, n_col, device=device)
            .view(-1, 1).requires_grad_(True) for i in range(3)]
zeta_iface = [torch.tensor([[bounds[1]]], device=device, requires_grad=True),
              torch.tensor([[bounds[2]]], device=device, requires_grad=True)]
sensor_zeta_t = torch.tensor(sensor_zeta, dtype=torch.float32, device=device).view(-1, 1)
sensor_u_t = torch.tensor(sensor_u, dtype=torch.float32, device=device).view(-1, 1)


def physics_residual(u_fn, E, zeta):
    """sigma(z) = E * du/dz = P0 (constante) -- forma integrada del equilibrio K(E)u=F, Eq. (12)."""
    u = u_fn(zeta)
    du = d_dz(u, zeta)
    return E * du - P0


def compute_loss_vanilla(model, E, lam_phy=1.0, lam_data=5.0):
    loss_phy = 0.0
    for i in range(3):
        res = physics_residual(model, E[i], zeta_dom[i])
        loss_phy = loss_phy + torch.mean(res**2)
    u_pred = model(sensor_zeta_t)
    loss_data = torch.mean((u_pred - sensor_u_t)**2)
    return lam_phy * loss_phy + lam_data * loss_data


def compute_loss_xpinn(model, E, lam_phy=1.0, lam_data=5.0):
    loss_phy = 0.0
    for i in range(3):
        fn = lambda z, i=i: model.forward_domain(z, i)
        res = physics_residual(fn, E[i], zeta_dom[i])
        loss_phy = loss_phy + torch.mean(res**2)

    # continuidad de interfaz (Eq. 14): desplazamiento predicho por subredes vecinas debe coincidir
    loss_int = 0.0
    for i, z_if in enumerate(zeta_iface):
        u_left = model.forward_domain(z_if, i)
        u_right = model.forward_domain(z_if, i + 1)
        loss_int = loss_int + torch.mean((u_left - u_right)**2)

    # dato: cada sensor se evalua con la subred de la capa a la que pertenece
    loss_data = 0.0
    for i in range(3):
        z0, z1 = bounds[i], bounds[i + 1]
        mask = (sensor_zeta >= z0) & (sensor_zeta <= z1)
        if mask.sum() == 0:
            continue
        z_s = sensor_zeta_t[mask]
        u_s = sensor_u_t[mask]
        u_pred = model.forward_domain(z_s, i)
        loss_data = loss_data + torch.sum((u_pred - u_s)**2)
    loss_data = loss_data / len(sensor_zeta)

    return lam_phy * (loss_phy + loss_int) + lam_data * loss_data

## 4. Entrenamiento: E se inicializa deliberadamente bajo (E=0.1, como en el paper, Seccion 3.4) y se aprende junto con los pesos de red

In [ ]:
def train(model_type, epochs=8000):
    # Adam con tasas de aprendizaje separadas para theta y para E (log-parametrizado):
    # E parte deliberadamente bajo (0.1, como en el paper, Seccion 3.4) y necesita pasos
    # mas grandes para alcanzar el rango fisico realista (cientos a miles de MPa).
    model = VanillaPINN().to(device) if model_type == 'vanilla' else XPINN().to(device)
    E_raw = torch.nn.Parameter(torch.log(torch.tensor([0.1, 0.1, 0.1])).to(device))
    opt = torch.optim.Adam([{'params': model.parameters(), 'lr': 1e-3},
                             {'params': [E_raw], 'lr': 0.05}])
    loss_fn = compute_loss_vanilla if model_type == 'vanilla' else compute_loss_xpinn
    history = []

    for epoch in range(epochs):
        opt.zero_grad()
        E = torch.exp(E_raw)
        loss = loss_fn(model, E)
        loss.backward()
        opt.step()
        history.append(loss.item())
        if epoch % 2000 == 0:
            print(f'[{model_type}] epoch {epoch:5d} | loss={loss.item():.3e} | '
                  f'E=[{E[0]:.1f}, {E[1]:.1f}, {E[2]:.1f}]  (verdad: {E_true})')

    return model, torch.exp(E_raw).detach().cpu().numpy(), history


model_vanilla, E_vanilla, hist_vanilla = train('vanilla')
model_xpinn, E_xpinn, hist_xpinn = train('xpinn')

## 5. Resultados: recuperacion de modulos (reproduce el hallazgo central del paper)

In [ ]:
print('Modulos verdaderos:      ', E_true)
print('PINN vanilla recupera:   ', np.round(E_vanilla, 1),
      ' -- error relativo:', np.round(np.abs(E_vanilla - E_true) / np.array(E_true) * 100, 1), '%')
print('XPINN recupera:          ', np.round(E_xpinn, 1),
      ' -- error relativo:', np.round(np.abs(E_xpinn - E_true) / np.array(E_true) * 100, 1), '%')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(3)
width = 0.25
axes[0].bar(x - width, E_true, width, label='Verdad')
axes[0].bar(x, E_vanilla, width, label='PINN vanilla')
axes[0].bar(x + width, E_xpinn, width, label='XPINN')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['E1 (subrasante)', 'E2 (base)', 'E3 (superficie)'])
axes[0].set_ylabel('Modulo (MPa)')
axes[0].set_title('Backcalculo de modulos: PINN vanilla vs XPINN')
axes[0].legend()

axes[1].semilogy(hist_vanilla, label='PINN vanilla')
axes[1].semilogy(hist_xpinn, label='XPINN')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss total (escala log)')
axes[1].set_title('Convergencia del entrenamiento')
axes[1].legend()
plt.tight_layout()
plt.show()

Con este entrenamiento (8000 epocas, Adam) se observa el mismo patron cualitativo que reporta el paper: la **PINN vanilla converge a un unico valor de $E$ practicamente identico para las tres capas** (es incapaz de representar la discontinuidad de material entre capas, quedandose cerca solo del valor de la capa que domina su ajuste), mientras que la **XPINN si logra diferenciar los tres modulos** entre si, aunque su convergencia hacia los valores exactos sigue siendo lenta e imperfecta en este numero de epocas -- exactamente el punto que subraya el paper: XPINN mejora sobre la PINN vanilla pero **sigue siendo sensible a la ponderacion de la perdida, la arquitectura y el numero de iteraciones**, muy lejos de la robustez de un solver FEM diferenciable con restricciones duras. Para el forward-model FEM axisimetrico 2D completo, el ajuste fino de hiperparametros, y la comparacion cuantitativa con DiffFEM, ver el paper original y el [repositorio de referencia de XPINN](https://github.com/AmeyaJagtap/XPINNs).